# Wildfire Data Preprocessing

This notebook is the Jupyter interface for the shared preprocessing pipeline. The implementation remains in `shared/preprocessing.py` so all model scripts use exactly the same behavior.

The pipeline loads TFRecords from `data/`, applies the fixed train/eval/test split, fills missing input values, normalizes inputs using training statistics, creates valid-label masks, and returns PyTorch DataLoaders.

In [ ]:
from pathlib import Path

In [ ]:
# Notebook-only shared preprocessing implementation

import glob
import json
import os
from dataclasses import asdict, dataclass
from typing import Dict, List, Tuple

import numpy as np
import tensorflow as tf
import torch
from torch.utils.data import DataLoader, Dataset

SEED = 42
PATCH_SIZE = 64
INPUT_FEATURES: List[str] = [
    "elevation", "th", "vs", "tmmn", "tmmx", "sph", "pr",
    "pdsi", "NDVI", "population", "erc", "PrevFireMask",
]
TARGET_FEATURE = "FireMask"
ALL_FEATURES = INPUT_FEATURES + [TARGET_FEATURE]
NUM_CHANNELS = len(INPUT_FEATURES)
UNCERTAIN_LABEL_VALUE = -1.0
WIND_DIRECTION_CHANNEL = INPUT_FEATURES.index("th")
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
STATS_PATH = PROJECT_ROOT / "shared" / "normalization_stats.json"

np.random.seed(SEED)
torch.manual_seed(SEED)


def _feature_description() -> Dict[str, tf.io.FixedLenFeature]:
    return {
        name: tf.io.FixedLenFeature([PATCH_SIZE, PATCH_SIZE], tf.float32)
        for name in ALL_FEATURES
    }


def _parse_example(example_proto):
    parsed = tf.io.parse_single_example(example_proto, _feature_description())
    inputs = tf.stack([parsed[name] for name in INPUT_FEATURES], axis=0)
    target = parsed[TARGET_FEATURE][tf.newaxis, ...]
    return inputs, target


def load_split_arrays(tfrecord_paths: List[str]) -> Tuple[np.ndarray, np.ndarray]:
    if not tfrecord_paths:
        raise FileNotFoundError(
            "No TFRecord files found. Put the dataset files in data/."
        )
    raw_dataset = tf.data.TFRecordDataset(tfrecord_paths)
    parsed_dataset = raw_dataset.map(_parse_example)
    all_inputs, all_targets = [], []
    for inputs, target in parsed_dataset:
        all_inputs.append(inputs.numpy())
        all_targets.append(target.numpy())
    if not all_inputs:
        raise ValueError("The TFRecord files contain 0 examples.")
    return np.stack(all_inputs), np.stack(all_targets)


def get_split_files(data_dir: str) -> Dict[str, List[str]]:
    splits = {
        "train": sorted(glob.glob(os.path.join(data_dir, "*train*.tfrecord*"))),
        "val": sorted(glob.glob(os.path.join(data_dir, "*eval*.tfrecord*"))),
        "test": sorted(glob.glob(os.path.join(data_dir, "*test*.tfrecord*"))),
    }
    for split_name, files in splits.items():
        if not files:
            raise FileNotFoundError(
                f"No files found for split '{split_name}' in '{data_dir}'."
            )
    return splits


def build_valid_mask(targets: np.ndarray) -> np.ndarray:
    return (targets != UNCERTAIN_LABEL_VALUE).astype(np.float32)


def _fill_nans_with_medians(inputs: np.ndarray, channel_medians: np.ndarray) -> np.ndarray:
    inputs = inputs.copy()
    for channel_index in range(inputs.shape[1]):
        channel = inputs[:, channel_index, :, :]
        nan_mask = np.isnan(channel)
        if nan_mask.any():
            channel[nan_mask] = channel_medians[channel_index]
            inputs[:, channel_index, :, :] = channel
    return inputs


@dataclass
class NormalizationStats:
    means: List[float]
    stds: List[float]
    medians: List[float]

    def save(self, path: str = str(STATS_PATH)) -> None:
        with open(path, "w") as file:
            json.dump(asdict(self), file, indent=2)

    @classmethod
    def load(cls, path: str = str(STATS_PATH)) -> "NormalizationStats":
        with open(path) as file:
            return cls(**json.load(file))


def compute_normalization_stats(train_inputs: np.ndarray) -> NormalizationStats:
    means, stds, medians = [], [], []
    for channel_index in range(train_inputs.shape[1]):
        channel = train_inputs[:, channel_index, :, :]
        median = float(np.nanmedian(channel))
        medians.append(median)
        clean_channel = np.nan_to_num(channel, nan=median)
        means.append(float(clean_channel.mean()))
        stds.append(float(clean_channel.std() + 1e-8))
    return NormalizationStats(means=means, stds=stds, medians=medians)


def normalize_inputs(inputs: np.ndarray, stats: NormalizationStats) -> np.ndarray:
    inputs = _fill_nans_with_medians(inputs, np.array(stats.medians))
    means = np.array(stats.means).reshape(1, -1, 1, 1)
    stds = np.array(stats.stds).reshape(1, -1, 1, 1)
    return (inputs - means) / stds


def compute_pos_weight(train_targets: np.ndarray) -> float:
    valid_labels = train_targets[build_valid_mask(train_targets).astype(bool)]
    num_pos = (valid_labels == 1).sum()
    num_neg = (valid_labels == 0).sum()
    if num_pos == 0:
        raise ValueError("No positive fire pixels found in training targets.")
    return float(num_neg / num_pos)


class WildfireDataset(Dataset):
    def __init__(self, inputs: np.ndarray, targets: np.ndarray, augment: bool = False):
        self.inputs = inputs
        self.targets = targets
        self.valid_mask = build_valid_mask(targets)
        self.augment = augment

    def __len__(self) -> int:
        return len(self.inputs)

    def __getitem__(self, index: int):
        inputs = self.inputs[index]
        targets = self.targets[index]
        valid_mask = self.valid_mask[index]
        if self.augment:
            inputs, targets, valid_mask = self._augment(inputs, targets, valid_mask)
        return (
            torch.from_numpy(inputs.copy()).float(),
            torch.from_numpy(targets.copy()).float(),
            torch.from_numpy(valid_mask.copy()).float(),
        )

    @staticmethod
    def _augment(inputs, targets, valid_mask):
        if np.random.rand() < 0.5:
            inputs = inputs[:, :, ::-1]
            targets = targets[:, :, ::-1]
            valid_mask = valid_mask[:, :, ::-1]
            inputs[WIND_DIRECTION_CHANNEL] = (180.0 - inputs[WIND_DIRECTION_CHANNEL]) % 360.0
        if np.random.rand() < 0.5:
            inputs = inputs[:, ::-1, :]
            targets = targets[:, ::-1, :]
            valid_mask = valid_mask[:, ::-1, :]
            inputs[WIND_DIRECTION_CHANNEL] = (360.0 - inputs[WIND_DIRECTION_CHANNEL]) % 360.0
        rotation = np.random.randint(0, 4)
        if rotation:
            inputs = np.rot90(inputs, k=rotation, axes=(1, 2))
            targets = np.rot90(targets, k=rotation, axes=(1, 2))
            valid_mask = np.rot90(valid_mask, k=rotation, axes=(1, 2))
            inputs[WIND_DIRECTION_CHANNEL] = (
                inputs[WIND_DIRECTION_CHANNEL] - 90.0 * rotation
            ) % 360.0
        return inputs, targets, valid_mask


def get_dataloaders(
    data_dir: str = "data",
    batch_size: int = 32,
    augment_train: bool = True,
    num_workers: int = 0,
):
    split_files = get_split_files(data_dir)
    train_inputs, train_targets = load_split_arrays(split_files["train"])
    val_inputs, val_targets = load_split_arrays(split_files["val"])
    test_inputs, test_targets = load_split_arrays(split_files["test"])

    if STATS_PATH.exists():
        stats = NormalizationStats.load()
    else:
        stats = compute_normalization_stats(train_inputs)
        stats.save()

    train_inputs = normalize_inputs(train_inputs, stats)
    val_inputs = normalize_inputs(val_inputs, stats)
    test_inputs = normalize_inputs(test_inputs, stats)
    pos_weight = compute_pos_weight(train_targets)

    generator = torch.Generator()
    generator.manual_seed(SEED)
    train_loader = DataLoader(
        WildfireDataset(train_inputs, train_targets, augment=augment_train),
        batch_size=batch_size, shuffle=True, num_workers=num_workers, generator=generator,
    )
    val_loader = DataLoader(
        WildfireDataset(val_inputs, val_targets), batch_size=batch_size, shuffle=False,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        WildfireDataset(test_inputs, test_targets), batch_size=batch_size, shuffle=False,
        num_workers=num_workers,
    )
    return train_loader, val_loader, test_loader, pos_weight

print("Shared notebook preprocessing functions are ready.")

## 1. Import the shared pipeline

Run this notebook from the repository root, or change `PROJECT_ROOT` to the project folder.

In [ ]:
print(f"Input channels: {NUM_CHANNELS}")
print(f"Patch size: {PATCH_SIZE} x {PATCH_SIZE}")
print(f"Features: {INPUT_FEATURES}")

## 2. Find the dataset splits

Put the downloaded `.tfrecord` files directly in `data/`. Their names must contain `train`, `eval`, and `test`.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
split_files = get_split_files(str(DATA_DIR))

for split_name, files in split_files.items():
    print(f"{split_name}: {len(files)} file(s)")
    for file_path in files:
        print(f"  {file_path}")

## 3. Load raw arrays

Each input sample has shape `(12, 64, 64)`. Each target has shape `(1, 64, 64)` and may contain `-1` for uncertain pixels.

In [ ]:
train_inputs, train_targets = load_split_arrays(split_files["train"])
val_inputs, val_targets = load_split_arrays(split_files["val"])
test_inputs, test_targets = load_split_arrays(split_files["test"])

print("Train:", train_inputs.shape, train_targets.shape)
print("Validation:", val_inputs.shape, val_targets.shape)
print("Test:", test_inputs.shape, test_targets.shape)

## 4. Normalize inputs and create masks

Statistics are computed from the training inputs only and reused for validation and test data. The mask marks valid target pixels.

In [ ]:
stats_path = PROJECT_ROOT / "shared" / "normalization_stats.json"
if stats_path.exists():
    stats = NormalizationStats.load(str(stats_path))
else:
    stats = compute_normalization_stats(train_inputs)
    stats.save(str(stats_path))

train_inputs = normalize_inputs(train_inputs, stats)
val_inputs = normalize_inputs(val_inputs, stats)
test_inputs = normalize_inputs(test_inputs, stats)

train_mask = build_valid_mask(train_targets)
val_mask = build_valid_mask(val_targets)
test_mask = build_valid_mask(test_targets)

print("Normalization statistics: ready")
print("Valid training pixels:", int(train_mask.sum()))

## 5. Create PyTorch DataLoaders

Models should use this function instead of rebuilding the pipeline. It returns `train_loader`, `val_loader`, `test_loader`, and the shared positive-class weight.

In [ ]:
train_loader, val_loader, test_loader, pos_weight = get_dataloaders(
    data_dir=str(DATA_DIR),
    batch_size=32,
    augment_train=True,
    num_workers=0,
)

print(f"Train batches: {len(train_loader)}")
print(f"Validation batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")
print(f"Positive-class weight: {pos_weight:.3f}")

## 6. Check one batch

A model receives `inputs`. Training loss and evaluation metrics must use `valid_mask` so uncertain target pixels do not contribute.

In [ ]:
inputs, targets, valid_mask = next(iter(train_loader))

print("Inputs:", tuple(inputs.shape), inputs.dtype)
print("Targets:", tuple(targets.shape), targets.dtype)
print("Valid mask:", tuple(valid_mask.shape), valid_mask.dtype)

assert inputs.shape[1:] == (NUM_CHANNELS, PATCH_SIZE, PATCH_SIZE)
assert targets.shape[1:] == (1, PATCH_SIZE, PATCH_SIZE)
assert valid_mask.shape == targets.shape
print("Preprocessing smoke test passed.")